# Melanoma Results: Confidence Intervals and Run Comparisons

This supplementary notebook analyses the saved outputs from the frozen Attention U-Net + EfficientNetB0 evaluation. It does **not** train models, change thresholds, or claim a new independent test. It adds image-level bootstrap intervals, Wilson intervals for proportions, a reproducible metric summary, and optional paired comparisons with earlier saved runs (including hair-removal experiments).

Run the latest reporting notebook first so the per-image CSV files exist in Google Drive. The current official test set has been inspected during project development, so all test-set intervals here quantify sampling uncertainty for this dataset; they do not remove reuse or selection bias.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, confusion_matrix, roc_curve
from IPython.display import display

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print('Google Colab Drive mount is unavailable. Set RUN_DIR to a local folder instead.')

## 1. Point to the saved evaluation run

Set `RUN_DIR` to the output folder printed by the latest reporting notebook. The default below matches the run path recorded in that notebook. If you ran it again, replace the path with the newer run folder.

In [ ]:
RUN_DIR = Path('/content/drive/MyDrive/Melanoma_Attention_UNet_Reporting_and_App/20261012T051542_296259Z')
TABLES_DIR = RUN_DIR / 'tables'
FIGURES_DIR = RUN_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

CLASSIFICATION_PREDICTIONS = TABLES_DIR / 'FOLLOWUP_official_test_predictions.csv'
SEGMENTATION_PER_IMAGE = TABLES_DIR / 'complete_segmentation_per_image.csv'
CLASSIFICATION_METRICS = TABLES_DIR / 'complete_classification_test_metrics.csv'
SEGMENTATION_METRICS = TABLES_DIR / 'complete_segmentation_test_metrics.csv'

required = [CLASSIFICATION_PREDICTIONS, SEGMENTATION_PER_IMAGE,
            CLASSIFICATION_METRICS, SEGMENTATION_METRICS]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError(
        'Required saved outputs are missing. Run the latest reporting notebook first, '
        'then check RUN_DIR. Missing files:\n' + '\n'.join(missing)
    )

cls = pd.read_csv(CLASSIFICATION_PREDICTIONS)
seg = pd.read_csv(SEGMENTATION_PER_IMAGE)
cls_summary = pd.read_csv(CLASSIFICATION_METRICS).iloc[0]
seg_summary = pd.read_csv(SEGMENTATION_METRICS).iloc[0]

required_cls_cols = {'image_id', 'label', 'malignancy_probability', 'predicted_label'}
required_seg_cols = {'image_id', 'dice', 'iou', 'test_loss', 'pixel_TP', 'pixel_TN', 'pixel_FP', 'pixel_FN'}
assert required_cls_cols.issubset(cls.columns), f'Classification CSV needs: {sorted(required_cls_cols)}'
assert required_seg_cols.issubset(seg.columns), f'Segmentation CSV needs: {sorted(required_seg_cols)}'
assert cls.image_id.is_unique and seg.image_id.is_unique, 'Duplicate image IDs found.'
assert set(cls.image_id) == set(seg.image_id), 'Classification and segmentation image IDs do not match.'

threshold = float(cls_summary['Threshold'])
if not np.array_equal(cls['predicted_label'].to_numpy(dtype=int),
                      (cls['malignancy_probability'].to_numpy() >= threshold).astype(int)):
    raise ValueError('Saved labels do not match the threshold recorded in the metric file.')

print(f'Loaded {len(cls)} paired test cases; threshold={threshold:.6f}.')
print(f'Melanoma={int(cls.label.sum())}; benign={int((cls.label == 0).sum())}.')
print('Saved segmentation checkpoint and threshold are not changed by this notebook.')

## 2. Classification results and 95% intervals

Accuracy, sensitivity, specificity and precision receive Wilson intervals. F1-score, balanced accuracy, ROC-AUC and unweighted binary cross-entropy receive stratified case-level bootstrap percentile intervals. Resampling is stratified by reference diagnosis so both classes remain represented in every replicate.

In [ ]:
BOOTSTRAP_REPLICATES = 5000
RANDOM_SEED = 20261107
rng = np.random.default_rng(RANDOM_SEED)
y = cls['label'].to_numpy(dtype=int)
prob = cls['malignancy_probability'].to_numpy(dtype=float)
pred = cls['predicted_label'].to_numpy(dtype=int)

def wilson_interval(successes, n, z=1.959963984540054):
    if n <= 0:
        return (np.nan, np.nan)
    p = successes / n
    den = 1 + z*z/n
    centre = (p + z*z/(2*n)) / den
    half = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n)) / den
    return centre-half, centre+half

def classification_stats(y_true, p_score, threshold):
    y_true = np.asarray(y_true, dtype=int)
    p_score = np.asarray(p_score, dtype=float)
    pred_label = (p_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred_label, labels=[0,1]).ravel()
    sens = tp/(tp+fn) if tp+fn else np.nan
    spec = tn/(tn+fp) if tn+fp else np.nan
    prec = tp/(tp+fp) if tp+fp else 0.0
    f1 = 2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.0
    clipped = np.clip(p_score, 1e-7, 1-1e-7)
    loss = np.mean(-y_true*np.log(clipped) - (1-y_true)*np.log(1-clipped))
    return {
        'Accuracy': (tp+tn)/len(y_true), 'Sensitivity': sens,
        'Specificity': spec, 'Precision': prec, 'F1-score': f1,
        'Balanced accuracy': (sens+spec)/2,
        'ROC-AUC': roc_auc_score(y_true, p_score),
        'Classification test loss (unweighted BCE)': loss,
        'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)
    }

point = classification_stats(y, prob, threshold)
idx0 = np.flatnonzero(y == 0)
idx1 = np.flatnonzero(y == 1)
boot_keys = ['Accuracy','Sensitivity','Specificity','Precision','F1-score',
             'Balanced accuracy','ROC-AUC','Classification test loss (unweighted BCE)']
boot = {key: [] for key in boot_keys}
for _ in range(BOOTSTRAP_REPLICATES):
    sample = np.concatenate([
        rng.choice(idx0, size=len(idx0), replace=True),
        rng.choice(idx1, size=len(idx1), replace=True)
    ])
    values = classification_stats(y[sample], prob[sample], threshold)
    for key in boot_keys:
        boot[key].append(values[key])

proportion_denominators = {
    'Accuracy': (point['TP']+point['TN'], len(y)),
    'Sensitivity': (point['TP'], point['TP']+point['FN']),
    'Specificity': (point['TN'], point['TN']+point['FP']),
    'Precision': (point['TP'], point['TP']+point['FP'])
}
rows=[]
for key in boot_keys:
    vals=np.asarray(boot[key],dtype=float)
    lo,hi=np.quantile(vals,[0.025,0.975])
    method='Stratified image-level percentile bootstrap'
    if key in proportion_denominators:
        lo,hi=wilson_interval(*proportion_denominators[key])
        method='Wilson score interval'
    rows.append({'Metric':key,'Estimate':point[key],'95% CI lower':lo,
                 '95% CI upper':hi,'Interval method':method})
classification_ci = pd.DataFrame(rows)
display(classification_ci.round(4))
classification_ci.to_csv(TABLES_DIR/'supplemental_classification_confidence_intervals.csv',index=False)
print('Confusion counts:', {k:point[k] for k in ['TP','TN','FP','FN']})

## 3. Segmentation results and image-level intervals

Bootstrap resampling is performed over images (not pixels), preserving the image as the sampling unit. Pixel sensitivity and specificity are recomputed from the per-image pixel counts within each bootstrap sample. This avoids treating millions of correlated pixels as independent observations.

In [ ]:
seg = seg.copy()
for col in ['dice','iou','test_loss','pixel_TP','pixel_TN','pixel_FP','pixel_FN']:
    seg[col] = pd.to_numeric(seg[col], errors='raise')
seg = seg.reset_index(drop=True)
N = len(seg)
if N < 2:
    raise ValueError('At least two per-image rows are required for bootstrap intervals.')

seg_values = {key: [] for key in ['Mean Dice','Median Dice','Mean IoU','Mean segmentation test loss',
                                   'Pooled pixel sensitivity','Pooled pixel specificity']}
for _ in range(BOOTSTRAP_REPLICATES):
    b = seg.iloc[rng.integers(0,N,size=N)]
    tp_px,tn_px,fp_px,fn_px = [b[c].sum() for c in ['pixel_TP','pixel_TN','pixel_FP','pixel_FN']]
    seg_values['Mean Dice'].append(b.dice.mean())
    seg_values['Median Dice'].append(b.dice.median())
    seg_values['Mean IoU'].append(b.iou.mean())
    seg_values['Mean segmentation test loss'].append(b.test_loss.mean())
    seg_values['Pooled pixel sensitivity'].append(tp_px/(tp_px+fn_px) if tp_px+fn_px else np.nan)
    seg_values['Pooled pixel specificity'].append(tn_px/(tn_px+fp_px) if tn_px+fp_px else np.nan)

point_seg = {
    'Mean Dice':seg.dice.mean(), 'Median Dice':seg.dice.median(),
    'Mean IoU':seg.iou.mean(), 'Mean segmentation test loss':seg.test_loss.mean(),
    'Pooled pixel sensitivity':seg.pixel_TP.sum()/(seg.pixel_TP.sum()+seg.pixel_FN.sum()),
    'Pooled pixel specificity':seg.pixel_TN.sum()/(seg.pixel_TN.sum()+seg.pixel_FP.sum())
}
seg_rows=[]
for key,val in point_seg.items():
    lo,hi=np.nanquantile(seg_values[key],[0.025,0.975])
    seg_rows.append({'Metric':key,'Estimate':val,'95% CI lower':lo,'95% CI upper':hi,
                     'Interval method':'Image-level percentile bootstrap'})
segmentation_ci=pd.DataFrame(seg_rows)
display(segmentation_ci.round(4))
segmentation_ci.to_csv(TABLES_DIR/'supplemental_segmentation_confidence_intervals.csv',index=False)
print(f'Images: {N}; zero-Dice cases: {(seg.dice == 0).sum()}; median Dice: {seg.dice.median():.4f}.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11,4.5))
axes[0].hist(seg.dice, bins=np.linspace(0,1,21), color='#2b7894', edgecolor='white')
axes[0].axvline(seg.dice.mean(), color='#c43c35', linestyle='--', label=f'Mean {seg.dice.mean():.3f}')
axes[0].axvline(seg.dice.median(), color='#2b5d3c', linestyle=':', label=f'Median {seg.dice.median():.3f}')
axes[0].set(xlabel='Per-image Dice', ylabel='Number of images', title='Segmentation Dice distribution')
axes[0].legend()
cm=confusion_matrix(y,pred,labels=[0,1])
axes[1].imshow(cm,cmap='Blues')
axes[1].set_xticks([0,1],labels=['Benign','Melanoma'])
axes[1].set_yticks([0,1],labels=['Benign','Melanoma'])
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('Reference')
axes[1].set_title(f'Classification at cutoff {threshold:.4f}')
for i in range(2):
    for j in range(2): axes[1].text(j,i,str(cm[i,j]),ha='center',va='center',fontsize=14)
fig.tight_layout()
fig.savefig(FIGURES_DIR/'supplemental_dice_and_confusion.png',dpi=220,bbox_inches='tight')
plt.show()

## 4. Optional paired comparisons with earlier runs

Use this section for saved hair-removal, augmentation, ROI, or architecture experiments. Set each run folder to a folder that contains the same standard output files under `tables/`. The notebook will calculate paired deltas only when both runs contain the same image IDs and the same diagnosis labels. Otherwise, it reports separate descriptive summaries and warns that the runs are not a paired comparison.

Do not compare a hair-removal run against the current 379-case run as if paired unless the exact image IDs match. If an older notebook saved only aggregate metrics, those values can be reported descriptively, but they cannot support paired confidence intervals.

In [ ]:
# Replace None with the saved run folders for experiments already completed.
# Example: 'hair_removal_on': Path('/content/drive/MyDrive/.../hair_on_run')
PRIOR_RUN_DIRS = {
    'hair_removal_on': None,
    'hair_removal_off': None,
    'augmentation_variant': None,
    'ROI_or_classifier_variant': None,
    'other_segmentation_pipeline': None,
}


def load_run(run_dir):
    t=Path(run_dir)/'tables'
    pred_file=t/'FOLLOWUP_official_test_predictions.csv'
    seg_file=t/'complete_segmentation_per_image.csv'
    if not pred_file.is_file() or not seg_file.is_file():
        raise FileNotFoundError(f'{run_dir} must contain {pred_file.name} and {seg_file.name} in tables/.')
    return pd.read_csv(pred_file),pd.read_csv(seg_file)

def summarize_run(name, cls_df, seg_df, threshold_value):
    m=classification_stats(cls_df.label.to_numpy(dtype=int),
                          cls_df.malignancy_probability.to_numpy(dtype=float),
                          float(threshold_value))
    return {'Run':name,'N classification':len(cls_df),'N segmentation':len(seg_df),
            'Mean Dice':seg_df.dice.mean(),'Mean IoU':seg_df.iou.mean(),
            'Accuracy':m['Accuracy'],'Sensitivity':m['Sensitivity'],
            'Specificity':m['Specificity'],'ROC-AUC':m['ROC-AUC'],
            'Threshold':float(threshold_value)}

def paired_bootstrap_deltas(base, variant, value_fn_base, value_fn_variant, keys, n_boot=BOOTSTRAP_REPLICATES):
    """Case-paired, class-stratified bootstrap; returns variant minus baseline."""
    base=base.reset_index(drop=True)
    variant=variant.reset_index(drop=True)
    labels=base.label.to_numpy(dtype=int)
    i0=np.flatnonzero(labels==0); i1=np.flatnonzero(labels==1)
    point_base=value_fn_base(base); point_variant=value_fn_variant(variant)
    samples={k:[] for k in keys}
    for _ in range(n_boot):
        ix=np.concatenate([rng.choice(i0,size=len(i0),replace=True),
                           rng.choice(i1,size=len(i1),replace=True)])
        vb=value_fn_base(base.iloc[ix]); vv=value_fn_variant(variant.iloc[ix])
        for k in keys: samples[k].append(vv[k]-vb[k])
    rows=[]
    for k in keys:
        d=point_variant[k]-point_base[k]
        lo,hi=np.quantile(samples[k],[.025,.975])
        rows.append({'Metric':k,'Variant minus baseline':d,'95% paired bootstrap CI lower':lo,
                     '95% paired bootstrap CI upper':hi})
    return rows

baseline_threshold=threshold
comparison_rows=[summarize_run('Current frozen Attention U-Net',cls,seg,baseline_threshold)]
paired_metric_rows=[]
for name,run_dir in PRIOR_RUN_DIRS.items():
    if run_dir is None:
        continue
    other_cls,other_seg=load_run(run_dir)
    metrics_path=Path(run_dir)/'tables'/'complete_classification_test_metrics.csv'
    other_threshold=float(pd.read_csv(metrics_path).iloc[0]['Threshold']) if metrics_path.is_file() else baseline_threshold
    comparison_rows.append(summarize_run(name,other_cls,other_seg,other_threshold))

    base_cls=cls.sort_values('image_id').reset_index(drop=True)
    var_cls=other_cls.sort_values('image_id').reset_index(drop=True)
    same_cls=(base_cls.image_id.tolist()==var_cls.image_id.tolist() and
              np.array_equal(base_cls.label.to_numpy(dtype=int),var_cls.label.to_numpy(dtype=int)))
    base_seg=seg.sort_values('image_id').reset_index(drop=True)
    var_seg=other_seg.sort_values('image_id').reset_index(drop=True)
    same_seg=base_seg.image_id.tolist()==var_seg.image_id.tolist()

    if same_cls:
        def base_cls_fn(frame):
            return classification_stats(frame.label.to_numpy(dtype=int),
                frame.malignancy_probability.to_numpy(dtype=float),baseline_threshold)
        def variant_cls_fn(frame):
            return classification_stats(frame.label.to_numpy(dtype=int),
                frame.malignancy_probability.to_numpy(dtype=float),other_threshold)
        paired_metric_rows.extend([{'Comparison':name+' vs current',**row} for row in paired_bootstrap_deltas(
            base_cls,var_cls,base_cls_fn,variant_cls_fn,['Accuracy','Sensitivity','Specificity','Precision','F1-score',
            'Balanced accuracy','ROC-AUC','Classification test loss (unweighted BCE)'])])
    else:
        print(f'{name}: classification test IDs differ from current run; report separately, do not calculate paired CIs.')

    if same_seg:
        def seg_fn(frame):
            tp_px=frame.pixel_TP.sum(); tn_px=frame.pixel_TN.sum()
            fp_px=frame.pixel_FP.sum(); fn_px=frame.pixel_FN.sum()
            return {'Mean Dice':frame.dice.mean(),'Mean IoU':frame.iou.mean(),
                    'Mean segmentation test loss':frame.test_loss.mean(),
                    'Pooled pixel sensitivity':tp_px/(tp_px+fn_px),
                    'Pooled pixel specificity':tn_px/(tn_px+fp_px)}
        # The paired image IDs are aligned before resampling. The same bootstrap sample
        # is applied to both runs, preserving the paired comparison.
        b=base_seg.reset_index(drop=True); v=var_seg.reset_index(drop=True)
        boot_seg={k:[] for k in seg_fn(b)}
        for _ in range(BOOTSTRAP_REPLICATES):
            ix=rng.integers(0,len(b),size=len(b))
            sb=seg_fn(b.iloc[ix]); sv=seg_fn(v.iloc[ix])
            for k in boot_seg: boot_seg[k].append(sv[k]-sb[k])
        sb=seg_fn(b); sv=seg_fn(v)
        for k in boot_seg:
            lo,hi=np.quantile(boot_seg[k],[.025,.975])
            paired_metric_rows.append({'Comparison':name+' vs current','Metric':k,
                'Variant minus baseline':sv[k]-sb[k],
                '95% paired bootstrap CI lower':lo,'95% paired bootstrap CI upper':hi})
    else:
        print(f'{name}: segmentation test IDs differ from current run; report separately, do not calculate paired CIs.')

comparison_summary=pd.DataFrame(comparison_rows)
display(comparison_summary)
comparison_summary.to_csv(TABLES_DIR/'supplemental_run_comparison_summary.csv',index=False)
if paired_metric_rows:
    paired_summary=pd.DataFrame(paired_metric_rows)
    display(paired_summary)
    paired_summary.to_csv(TABLES_DIR/'supplemental_paired_comparison_deltas.csv',index=False)
else:
    print('No paired comparison outputs produced. Configure prior run folders and confirm saved per-image CSVs.')


## 5. Training-curve and software-evidence provenance

This check reports what the saved run recorded. It does not reconstruct or certify training curves. If exact per-epoch CSVs are absent and the provenance file says values came from rounded historical console summaries, label those plots as historical and do not use them to explain the current checkpoint's performance.

In [ ]:
provenance_path=TABLES_DIR/'training_history_provenance.json'
if provenance_path.is_file():
    provenance=json.loads(provenance_path.read_text())
    print('Training-history mode:',provenance.get('mode'))
    print('Source run:',provenance.get('source_run'))
    print('Provenance note:',provenance.get('note'))
else:
    print('No training-history provenance file found.')
for filename in ['segmentation_training_history.csv','classifier_head_training_history.csv','classifier_fine_training_history.csv']:
    f=TABLES_DIR/filename
    print(f'{filename}:', 'exact CSV available' if f.is_file() else 'not available in this run folder')
parity=TABLES_DIR/'app_notebook_parity.csv'
if parity.is_file():
    parity_df=pd.read_csv(parity)
    print(f'Notebook–app parity cases recorded: {len(parity_df)}')
    display(parity_df)
else:
    print('No app_notebook_parity.csv found; do not claim parity testing from this run.')

## 6. Files created and reporting notes

The notebook saves the classification and segmentation interval tables, optional run comparison tables, and a supplementary Dice/confusion figure under the saved run folder. Use these as Results evidence only with their interval methods stated. The intervals do not validate the classifier clinically, repair segmentation failures, or undo prior use of the official test set.

In [ ]:
print('Supplemental files saved under:', RUN_DIR)
for p in sorted(TABLES_DIR.glob('supplemental_*')):
    print('-',p.name)
for p in sorted(FIGURES_DIR.glob('supplemental_*')):
    print('-',p.name)

## 7. Draft a Results paragraph from the calculated outputs

This cell creates a paste-ready paragraph using the computed intervals and observed counts. Check that the run description and test-set reuse caveat match the report before inserting it.

In [ ]:
class_ci=classification_ci.set_index('Metric')
seg_ci=segmentation_ci.set_index('Metric')
def ci_text(frame,metric,digits=3):
    row=frame.loc[metric]
    return f"{row['Estimate']:.{digits}f} (95% CI {row['95% CI lower']:.{digits}f}–{row['95% CI upper']:.{digits}f})"
def pct_text(frame,metric):
    row=frame.loc[metric]
    return f"{100*row['Estimate']:.1f}% (95% CI {100*row['95% CI lower']:.1f}–{100*row['95% CI upper']:.1f}%)"

n_cases=len(cls); n_melanoma=int(y.sum()); n_benign=int((y==0).sum())
tp,tn,fp,fn=[point[k] for k in ['TP','TN','FP','FN']]
seg_time=float(seg_summary.get('Evaluation seconds',np.nan))
text=(
 f"On the {n_cases}-image ISIC 2016 Task 3B evaluation set ({n_melanoma} melanoma and {n_benign} benign images), "
 f"the frozen Attention U-Net achieved a mean per-image Dice of {ci_text(seg_ci,'Mean Dice')} and mean IoU of {ci_text(seg_ci,'Mean IoU')}. "
 f"Median Dice was {seg.dice.median():.4f}, and {(seg.dice==0).sum()} image(s) had Dice 0. "
 f"At the validation-selected classifier cutoff of {threshold:.4f}, the model produced {int(tp)} true positives, {int(tn)} true negatives, {int(fp)} false positives and {int(fn)} false negatives. "
 f"Accuracy was {pct_text(class_ci,'Accuracy')}, sensitivity {pct_text(class_ci,'Sensitivity')}, specificity {pct_text(class_ci,'Specificity')}, "
 f"and precision {pct_text(class_ci,'Precision')}; ROC-AUC was {ci_text(class_ci,'ROC-AUC')}. "
 f"Classification test loss (unweighted binary cross-entropy) was {ci_text(class_ci,'Classification test loss (unweighted BCE)',4)}. "
 f"These are exploratory follow-up results because the official test set had been inspected during earlier development; they do not constitute independent clinical validation."
)
print(text)
(Tables_path:=TABLES_DIR/'supplemental_results_text.txt').write_text(text+'\n')
print('Saved:',Tables_path)